# NOORIX Master AI Training Orchestrator
**Prepared by Muhammad Farhan · NOORIX-AI**

This notebook runs the existing specialist training notebooks as independent stages, collects their output ZIPs, creates one ZIP per stage, and builds a combined release archive. A failed stage is recorded and does not stop later stages.

**Important:** This orchestrator does not make datasets appear automatically inside Kaggle. Attach the correct Kaggle datasets in the Input panel, or provide a verified downloadable source. Dataset licenses/access terms must be checked. Run one small stage first before enabling all stages. YOLO export needs a separate NOORIX adapter; Urdu OCR and ASR require labelled manifests.

In [ ]:
from pathlib import Path
import os, sys, json, re, shutil, zipfile, hashlib, subprocess, time, traceback
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'NOORIX-AI-training-source'
OUT = WORK / 'NOORIX_MASTER_OUTPUT'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Working directory:', WORK)
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e: print('PyTorch check:', e)


## 1. Configure the stages
Set paths to files under `/kaggle/input/...`. OCR and ASR require exact labels/transcriptions. PKR needs class folders named by denomination. YOLO custom training requires a valid `data.yaml`.

In [ ]:
# Start with one stage. Change RUN flags to True only after checking its dataset.
CONFIG = {
    'printed_english_ocr': {'run': False, 'annotation_csv': '', 'image_col': '', 'text_col': ''},
    'handwritten_english_ocr': {'run': False, 'annotation_csv': '', 'image_col': '', 'text_col': ''},
    'urdu_line_ocr': {'run': False, 'csv_path': '', 'image_col': 'image_path', 'text_col': 'text'},
    'pkr_currency': {'run': False, 'data_dir': ''},
    'yolo_objects_person': {'run': False, 'custom_data_yaml': '', 'test_image': ''},
    'whisper_urdu_asr': {'run': False, 'csv_path': '', 'audio_col': 'audio_path', 'text_col': 'text', 'language': 'urdu'},
    'whisper_english_asr': {'run': False, 'csv_path': '', 'audio_col': 'audio_path', 'text_col': 'text', 'language': 'english'},
}
GITHUB_BRANCH = 'feature/kaggle-ocr-training-notebooks'
GITHUB_REPO = 'https://github.com/Farhanillahiclass/NOORIX-AI.git'
MAX_NOTEBOOK_TIMEOUT = None  # None = no fixed timeout; monitor Kaggle GPU quota.
print('Enabled stages:', [k for k,v in CONFIG.items() if v['run']])
print('Kaggle input roots:')
for p in sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []: print(' -', p)


## 2. Fetch the specialist notebooks and validate configuration
The source notebooks remain the model-specific implementations. This controller patches only the documented path/language settings in temporary copies; it does not modify the GitHub source files.

In [ ]:
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',GITHUB_BRANCH,GITHUB_REPO,str(REPO)],check=True)
else:
    print('Using existing source checkout:', REPO)
NOTEBOOKS = {
 'printed_english_ocr': REPO/'notebooks/ocr/printed_ocr_training_muhammad_farhan.ipynb',
 'handwritten_english_ocr': REPO/'notebooks/ocr/handwritten_ocr_training_muhammad_farhan.ipynb',
 'urdu_line_ocr': REPO/'notebooks/ocr/urdu_printed_handwritten_crnn_ctc_muhammad_farhan.ipynb',
 'pkr_currency': REPO/'notebooks/currency/pkr_mobilenetv2_training_muhammad_farhan.ipynb',
 'yolo_objects_person': REPO/'notebooks/detection/yolov8n_objects_person_muhammad_farhan.ipynb',
 'whisper_urdu_asr': REPO/'notebooks/speech/whisper_urdu_english_asr_muhammad_farhan.ipynb',
 'whisper_english_asr': REPO/'notebooks/speech/whisper_urdu_english_asr_muhammad_farhan.ipynb',
}
def required_path(value, name, must_exist=True):
    if not value: raise ValueError(f'{name} is empty. Set its path in CONFIG.')
    p=Path(value)
    if must_exist and not p.exists(): raise FileNotFoundError(f'{name} does not exist: {p}')
    return str(p)
for key, cfg in CONFIG.items():
    if not cfg['run']: continue
    if key in ('printed_english_ocr','handwritten_english_ocr','urdu_line_ocr','whisper_urdu_asr','whisper_english_asr'):
        k='csv_path' if 'csv_path' in cfg else 'annotation_csv'
        required_path(cfg[k], f'{key}.{k}')
    elif key=='pkr_currency': required_path(cfg['data_dir'], 'pkr_currency.data_dir')
    elif key=='yolo_objects_person' and cfg['custom_data_yaml']: required_path(cfg['custom_data_yaml'], 'yolo.custom_data_yaml')
print('Configuration checks passed.')


## 3. Run stages independently
If a stage fails, its error is saved and the controller continues. For each stage, the corresponding original notebook is executed in a separate kernel.

In [ ]:
!pip -q install nbclient nbformat
import nbformat
from nbclient import NotebookClient
from copy import deepcopy
def set_assignment(nb, variable, value):
    # Replace a simple top-level assignment such as DATA_DIR='' or LANGUAGE='urdu'.
    pattern=re.compile(rf'(?m)^(\s*{re.escape(variable)}\s*=\s*).*$')
    replacement=f'{variable} = {json.dumps(value, ensure_ascii=False)}'
    hits=0
    for cell in nb.cells:
        if cell.cell_type!='code': continue
        source=cell.source
        source2,n=pattern.subn(lambda m: replacement, source)
        if n: cell.source=source2; hits+=n
    if hits==0: raise RuntimeError(f'Could not locate assignment for {variable!r} in notebook.')
def run_stage(name, cfg):
    src=NOTEBOOKS[name]
    if not src.is_file(): raise FileNotFoundError(src)
    nb=nbformat.read(src,as_version=4)
    if name in ('printed_english_ocr','handwritten_english_ocr'):
        set_assignment(nb,'ANNOTATION_CSV',cfg['annotation_csv'])
        set_assignment(nb,'IMAGE_COL',cfg.get('image_col',''))
        set_assignment(nb,'TEXT_COL',cfg.get('text_col',''))
    elif name=='urdu_line_ocr':
        set_assignment(nb,'CSV_PATH',cfg['csv_path'])
        set_assignment(nb,'IMAGE_COL',cfg.get('image_col','image_path'))
        set_assignment(nb,'TEXT_COL',cfg.get('text_col','text'))
    elif name=='pkr_currency': set_assignment(nb,'DATA_DIR',cfg['data_dir'])
    elif name=='yolo_objects_person':
        set_assignment(nb,'CUSTOM_DATA_YAML',cfg.get('custom_data_yaml',''))
        set_assignment(nb,'TEST_IMAGE',cfg.get('test_image',''))
    elif name in ('whisper_urdu_asr','whisper_english_asr'):
        set_assignment(nb,'CSV_PATH',cfg['csv_path'])
        set_assignment(nb,'AUDIO_COL',cfg.get('audio_col','audio_path'))
        set_assignment(nb,'TEXT_COL',cfg.get('text_col','text'))
        set_assignment(nb,'LANGUAGE',cfg['language'])
    temp=OUT/f'{name}_executed.ipynb'
    client=NotebookClient(nb,timeout=MAX_NOTEBOOK_TIMEOUT,kernel_name='python3',allow_errors=False,resources={'metadata':{'path':str(WORK)}})
    executed=client.execute()
    nbformat.write(executed,temp)
    return temp
results=[]
for name,cfg in CONFIG.items():
    if not cfg['run']: continue
    print('\n'+'='*72+'\nSTART:',name)
    start=time.time()
    try:
        executed=run_stage(name,cfg)
        results.append({'stage':name,'status':'success','seconds':round(time.time()-start,1),'executed_notebook':str(executed)})
        print('STAGE SUCCESS:',name)
    except Exception as exc:
        details=traceback.format_exc()
        (OUT/f'{name}_ERROR.txt').write_text(details,encoding='utf-8')
        results.append({'stage':name,'status':'failed','seconds':round(time.time()-start,1),'error':str(exc)})
        print('STAGE FAILED; continuing:',name,'|',repr(exc))
print('\nStage summary:',json.dumps(results,indent=2,ensure_ascii=False))


## 4. Package individual stage ZIPs and the combined release
Only outputs actually produced by a stage are included. Failed or disabled stages are never represented as trained models.

In [ ]:
# Find the output directories produced by specialist notebooks.
OUTPUT_PREFIXES = {
 'printed_english_ocr':'NOORIX_PRINTED_OCR',
 'handwritten_english_ocr':'NOORIX_HANDWRITTEN_OCR',
 'urdu_line_ocr':'NOORIX_Urdu_Line_OCR',
 'pkr_currency':'NOORIX_PKR_MobileNetV2',
 'yolo_objects_person':'NOORIX_Object_Person_YOLOv8n',
 'whisper_urdu_asr':'NOORIX_ASR_Whisper_urdu',
 'whisper_english_asr':'NOORIX_ASR_Whisper_english',
}
stage_archives=[]
for item in results:
    if item['status']!='success': continue
    prefix=OUTPUT_PREFIXES[item['stage']]
    candidates=[p for p in WORK.glob(prefix+'*.zip') if p.is_file()]
    if candidates:
        src=max(candidates,key=lambda p:p.stat().st_mtime)
        dest=OUT/f'{item["stage"]}.zip'
        shutil.copy2(src,dest)
        stage_archives.append(dest)
        item['artifact']=str(dest)
    else:
        item['status']='success_no_archive_found'
        print('No matching model archive found for:',item['stage'])
manifest={'project':'NOORIX-AI','prepared_by':'Muhammad Farhan','branch':GITHUB_BRANCH,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'stages':results,'warning':'A ZIP is not proof of model quality. Validate metrics, labels, preprocessing, loadability, and CPU latency before integration.'}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
combined=WORK/'NOORIX_AI_ALL_MODELS.zip'
with zipfile.ZipFile(combined,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in stage_archives: z.write(p,arcname=f'models/{p.name}')
    z.write(OUT/'manifest.json',arcname='manifest.json')
checks=[]
for p in [*stage_archives,OUT/'manifest.json']:
    digest=hashlib.sha256(p.read_bytes()).hexdigest()
    checks.append(f'{digest}  {p.name}')
(OUT/'checksums.sha256').write_text('\n'.join(checks)+'\n',encoding='utf-8')
with zipfile.ZipFile(combined,'a',compression=zipfile.ZIP_DEFLATED) as z: z.write(OUT/'checksums.sha256',arcname='checksums.sha256')
print('Stage ZIPs:',[str(p) for p in stage_archives])
print('Combined archive:',combined,'| bytes:',combined.stat().st_size)
print('Manifest:',OUT/'manifest.json')
print('No trained model is claimed for a failed/disabled stage.')


## 5. Integration warnings
- PKR classifier estimates denomination/side only; it does not authenticate notes or detect counterfeits.
- YOLOv8 ONNX output is not compatible with the current SSD adapter; a YOLO-specific decoder is required.
- Urdu CRNN/CTC is an experimental line-recognition baseline; assess CER on a separate test set.
- Whisper fine-tuning is GPU/memory intensive; validate WER/CER and latency before local CPU deployment.
- TrOCR English checkpoints are not Urdu OCR models.
- TTS is not trained here: NOORIX currently uses installed operating-system voices. ASR and TTS are different tasks.
- A stage marked successful means its notebook executed and generated output; it does not mean production quality or successful NOORIX integration.

In [ ]:
print(json.dumps(manifest,indent=2,ensure_ascii=False))
